# 02. Variables externas e ingeniería de características

Integra TRM y ONI mensuales. Todas las variables observadas se desplazan un mes para impedir fuga temporal.

In [1]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Proyecto_Buenaventura_Final')
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'src' else Path.cwd().resolve()

os.environ['PROYECTO_BUENAVENTURA_ROOT'] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT / 'src')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Proyecto:', PROJECT_ROOT)

Proyecto: C:\Users\juanc\Desktop\Proyecto_Buenaventura_Final


In [2]:
from external_data import build_external
from modeling import build_feature_table

ACTUALIZAR_EXTERNAS = False
externas = build_external(refresh=ACTUALIZAR_EXTERNAS)
features = build_feature_table(append_next=False)
print('Variables externas:', list(externas.columns))
features.tail()

Variables externas: ['trm_cop_usd', 'trm_volatilidad', 'oni_anomalia', 'enso_fase']


,cif_usd,peso_neto_kg,cif_usd_lag_1,cif_usd_lag_2,cif_usd_lag_3,cif_usd_lag_6,cif_usd_lag_12,cif_usd_media_3,cif_usd_media_12,peso_neto_kg_lag_1,...,seguros_usd_lag_1,registros_lag_1,paises_origen_lag_1,capitulos_lag_1,tendencia,mes_sin,mes_cos,trm_cop_usd_lag_1,trm_volatilidad_lag_1,oni_anomalia_lag_1
fecha,,,,,,,,,,,,,,,,,,,,,
2026-01-01,1.847331e+09,1.332863e+09,1.817764e+09,1.678903e+09,1.853764e+09,1.948659e+09,1.571568e+09,1.783477e+09,1.713634e+09,1.310585e+09,...,2591607.69,68658.0,83.0,94.0,168,0.500000,8.660254e-01,3797.459000,51.919655,-0.54
2026-02-01,1.811953e+09,1.218746e+09,1.847331e+09,1.817764e+09,1.678903e+09,1.780029e+09,1.643568e+09,1.781333e+09,1.736614e+09,1.332863e+09,...,2541596.63,73622.0,83.0,95.0,169,0.866025,5.000000e-01,3693.162632,44.151553,-0.37
2026-03-01,1.896815e+09,1.249995e+09,1.811953e+09,1.847331e+09,1.817764e+09,1.757864e+09,1.490017e+09,1.825683e+09,1.750647e+09,1.218746e+09,...,2922303.53,61708.0,84.0,93.0,170,1.000000,6.123234e-17,3678.698421,36.225841,-0.14
2026-04-01,1.940605e+09,1.352642e+09,1.896815e+09,1.811953e+09,1.847331e+09,1.853764e+09,1.631425e+09,1.852033e+09,1.784546e+09,1.249995e+09,...,3005625.00,62495.0,82.0,92.0,171,0.866025,-5.000000e-01,3714.856500,40.773840,0.13
2026-05-01,1.980453e+09,1.151371e+09,1.940605e+09,1.896815e+09,1.811953e+09,1.678903e+09,1.803485e+09,1.883124e+09,1.810311e+09,1.352642e+09,...,2476739.01,58484.0,83.0,93.0,172,0.500000,-8.660254e-01,3613.536000,39.132851,0.51


In [3]:
targets = {'cif_usd', 'peso_neto_kg'}
predictoras = [c for c in features.columns if c not in targets]
contemporaneas = {'fob_usd', 'flete_usd', 'seguros_usd', 'registros', 'paises_origen', 'capitulos'}
assert not contemporaneas.intersection(predictoras)
assert all(c in {'tendencia', 'mes_sin', 'mes_cos'} or '_lag_' in c or '_media_' in c for c in predictoras)
print('Control de fuga temporal aprobado:', len(predictoras), 'predictoras')

Control de fuga temporal aprobado: 26 predictoras
